In [1]:
import pandas as pd
import numpy as np

import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.models import load_model
import streamlit as st




In [2]:
file = "C:/Users/jlevi/Desktop/UNI/ITNPA4- 14/research_abstracts.json"
df = pd.read_json(file)

#drop duplicates
df = df.drop_duplicates()
duplicated_sum = df.duplicated().sum()
print(duplicated_sum)

#check for missing values
df = df.dropna()
print(df.isnull().sum())

# dropping uneeded column
df = df.drop(columns =['DOI'])
df.head()

0
DOI         0
Abstract    0
Label       0
dtype: int64


,Abstract,Label
0,Forward-looking forecasting of the inflation r...,ai
1,Our study examined the disaggregation of infla...,ai
2,This paper develops a dynamic factor model tha...,classical
3,"In this paper, I use a simple SIR Macro model ...",classical
4,BackgroundGame theory is useful for identifyin...,classical


In [3]:
#remove stopwords

stop_words = set(stopwords.words("english"))

df['Abstract']=df['Abstract'].apply( lambda x: " ".join(word for word in x.split() if word not in stop_words))

#make info lowercase
df['Abstract']=df['Abstract'].str.lower()

#remove punctuation 
df['Abstract'] =df['Abstract'].apply(lambda x: re.sub(r'[^\w\s]','',x))

#lemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

df["Abstract"] = df["Abstract"].apply(
    lambda x: " ".join(lemmatizer.lemmatize(word) for word in x.split()))

#Tokenise
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["Abstract"])
X_sequences = tokenizer.texts_to_sequences(df["Abstract"])

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(df["Label"])

# Make sure the data is numerical
X_padded = pad_sequences(X_sequences)
X_padded = np.array(X_padded, dtype="int32")
y_encoded = np.array(y_encoded, dtype="int32")

In [4]:
X = df["Abstract"]
y = df["Label"]

In [5]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(X)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

#Logistic Regression
model = LogisticRegression()
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
print("Testing Accuracy:", accuracy_score(y_test, y_test_pred))
print("Classification Report:")
print(classification_report(y_test, y_test_pred))

Training Accuracy: 0.8875413450937155
Testing Accuracy: 0.8722466960352423
Classification Report:
              precision    recall  f1-score   support

          ai       1.00      0.33      0.49        43
   classical       0.86      1.00      0.93       184

    accuracy                           0.87       227
   macro avg       0.93      0.66      0.71       227
weighted avg       0.89      0.87      0.84       227



In [7]:
# Train SVM
model_svm = SVC()
model_svm.fit(X_train, y_train)

# Make predictions
y_train_pred = model_svm.predict(X_train)
y_test_pred = model_svm.predict(X_test)

# Evaluate the model
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
print("Testing Accuracy:", accuracy_score(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Training Accuracy: 0.9878721058434399
Testing Accuracy: 0.8854625550660793

Classification Report:
              precision    recall  f1-score   support

          ai       1.00      0.40      0.57        43
   classical       0.88      1.00      0.93       184

    accuracy                           0.89       227
   macro avg       0.94      0.70      0.75       227
weighted avg       0.90      0.89      0.86       227



In [8]:
#data for LSTM
X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm = train_test_split(X_padded,y_encoded,test_size=0.2,random_state=42)

#LSTM
model_lstm = Sequential()
model_lstm.add(Embedding(input_dim=len(tokenizer.word_index) + 1,output_dim=100))
model_lstm.add(LSTM(64))
model_lstm.add(Dense(1, activation="sigmoid"))
model_lstm.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

history_lstm = model_lstm.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=10,
    validation_data=(X_test_lstm, y_test_lstm)
)

train_loss, train_accuracy = model_lstm.evaluate( X_train_lstm, y_train_lstm, verbose=0)
test_loss, test_accuracy = model_lstm.evaluate(X_test_lstm, y_test_lstm, verbose=0)
print("Training Accuracy:", train_accuracy)
print("Testing Accuracy:", test_accuracy)

Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 55s 2s/step - accuracy: 0.7660 - loss: 0.6320 - val_accuracy: 0.8106 - val_loss: 0.4619
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - accuracy: 0.7922 - loss: 0.4072 - val_accuracy: 0.8987 - val_loss: 0.3057
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 31s 1s/step - accuracy: 0.9270 - loss: 0.2165 - val_accuracy: 0.8943 - val_loss: 0.2837
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 27s 923ms/step - accuracy: 0.9764 - loss: 0.0954 - val_accuracy: 0.8987 - val_loss: 0.2751
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 23s 767ms/step - accuracy: 0.9997 - loss: 0.0295 - val_accuracy: 0.8943 - val_loss: 0.3508
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 26s 891ms/step - accuracy: 0.9986 - loss: 0.0133 - val_accuracy: 0.8678 - val_loss: 0.3938
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 26s 879ms/step - accuracy: 1.0000 - loss: 0.0068 - val_accuracy: 0.8767 - val_loss: 0.4235
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 22s 770ms/step - accuracy: 1.0000 - loss: 0.0039 - val_accuracy: 0.8

In [9]:
#Bi-LSTM
model_bilstm = Sequential()
model_bilstm.add(Embedding(input_dim=len(tokenizer.word_index) + 1,output_dim=100))
model_bilstm.add(Bidirectional(LSTM(64)))
model_bilstm.add(Dense(1, activation="sigmoid"))
model_bilstm.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

history_bilstm = model_bilstm.fit(X_train_lstm,y_train_lstm,epochs=10,validation_data=(X_test_lstm, y_test_lstm))

train_loss, train_accuracy = model_bilstm.evaluate(X_train_lstm, y_train_lstm, verbose=0)
test_loss, test_accuracy = model_bilstm.evaluate(X_test_lstm, y_test_lstm, verbose=0)
print("Training Accuracy:", train_accuracy)
print("Testing Accuracy:", test_accuracy)

Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.7243 - loss: 0.5830 - val_accuracy: 0.8106 - val_loss: 0.4266
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 57s 2s/step - accuracy: 0.8168 - loss: 0.3677 - val_accuracy: 0.9031 - val_loss: 0.2957
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 51s 2s/step - accuracy: 0.9661 - loss: 0.1608 - val_accuracy: 0.8767 - val_loss: 0.4254
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - accuracy: 0.9728 - loss: 0.0946 - val_accuracy: 0.8899 - val_loss: 0.2913
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 74s 1s/step - accuracy: 1.0000 - loss: 0.0198 - val_accuracy: 0.8943 - val_loss: 0.3663
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 42s 1s/step - accuracy: 1.0000 - loss: 0.0086 - val_accuracy: 0.8899 - val_loss: 0.4592
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step - accuracy: 1.0000 - loss: 0.0049 - val_accuracy: 0.8943 - val_loss: 0.4736
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 109s 3s/step - accuracy: 1.0000 - loss: 0.0036 - val_accuracy: 0.8811 - val_loss

In [10]:
#Q2.2 enhancement

from tensorflow.keras.layers import Dropout
model_lstm_dropout = Sequential()
model_lstm_dropout.add(Embedding(input_dim=len(tokenizer.word_index) + 1,output_dim=100))
model_lstm_dropout.add(LSTM(64))
model_lstm_dropout.add(Dropout(0.5))
model_lstm_dropout.add(Dense(1, activation="sigmoid"))
model_lstm_dropout.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history_dropout = model_lstm_dropout.fit(X_train_lstm,y_train_lstm,epochs=10,validation_data=(X_test_lstm, y_test_lstm))

train_loss, train_accuracy = model_lstm_dropout.evaluate(X_train_lstm, y_train_lstm, verbose=0)
test_loss, test_accuracy = model_lstm_dropout.evaluate(X_test_lstm, y_test_lstm, verbose=0)
print("Training Accuracy:", train_accuracy)
print("Testing Accuracy:", test_accuracy)

Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 33s 990ms/step - accuracy: 0.7491 - loss: 0.6282 - val_accuracy: 0.8150 - val_loss: 0.4353
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 32s 1s/step - accuracy: 0.7863 - loss: 0.4048 - val_accuracy: 0.8987 - val_loss: 0.2941
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 28s 969ms/step - accuracy: 0.9555 - loss: 0.1774 - val_accuracy: 0.8899 - val_loss: 0.2662
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.9857 - loss: 0.0520 - val_accuracy: 0.9383 - val_loss: 0.2545
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 35s 1s/step - accuracy: 0.9989 - loss: 0.0168 - val_accuracy: 0.9339 - val_loss: 0.2798
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 28s 972ms/step - accuracy: 0.9992 - loss: 0.0090 - val_accuracy: 0.9075 - val_loss: 0.3235
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.9958 - loss: 0.0276 - val_accuracy: 0.9075 - val_loss: 0.3661
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9975 - loss: 0.0119 - val_accuracy: 0.9163 - 

In [11]:
history_dropout = model_lstm_dropout.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=10,
    validation_data=(X_test_lstm, y_test_lstm)
)

model_lstm_dropout.save("researchly_model.keras")

import pickle

with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

with open("encoder.pkl", "wb") as file:
    pickle.dump(encoder, file)

max_len = X_padded.shape[1]

with open("max_len.pkl", "wb") as file:
    pickle.dump(max_len, file)
import os

print(os.listdir())


Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 1.0000 - loss: 0.0018 - val_accuracy: 0.9339 - val_loss: 0.3490
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - accuracy: 1.0000 - loss: 0.0013 - val_accuracy: 0.9339 - val_loss: 0.3434
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.9471 - val_loss: 0.3450
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 1.0000 - loss: 9.0762e-04 - val_accuracy: 0.9383 - val_loss: 0.3634
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 1.0000 - loss: 6.8387e-04 - val_accuracy: 0.9339 - val_loss: 0.3677
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 1.0000 - loss: 6.5917e-04 - val_accuracy: 0.9339 - val_loss: 0.3658
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 1.0000 - loss: 5.5290e-04 - val_accuracy: 0.9339 - val_loss: 0.3775
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 51s 2s/step - accuracy: 1.0000 - loss: 5.4758e-04 - val_accuracy

In [12]:
import os
print(os.getcwd())

C:\Users\jlevi


In [13]:
%%writefile app.py

import streamlit as st
import re
import pickle
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences


Overwriting app.py


In [14]:
# Load model and files
model = load_model("researchly_model.keras")
with open("tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)
with open("encoder.pkl", "rb") as file:
    encoder = pickle.load(file)
with open("max_len.pkl", "rb") as file:
    max_len = pickle.load(file)

def preprocess_text(text):
    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
    text = text.lower()
    text = " ".join(
        word for word in text.split()
        if word not in stop_words)
    text = " ".join(
        lemmatizer.lemmatize(word)
        for word in text.split())
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=max_len)

    return padded


In [15]:
#heading
st.title("Researchly Research Abstract Classifier")
st.write("This app classifies research abstracts as either AI related research or classical research")

#input
abstract = st.text_area("Enter research abstract:")

#button
if st.button("Classify Abstract"):

        processed_text = preprocess_text(abstract)
        prediction = model.predict(
            processed_text,
            verbose=0)
        predicted_class = int(prediction[0][0] >= 0.5)
        predicted_label = encoder.inverse_transform(
            [predicted_class]
        )[0]
        st.subheader("Prediction")

        st.write(
            "The abstract is classified as:",
            predicted_label
        )


2026-11-11 16:04:15.274 
  command:

    streamlit run C:\Users\jlevi\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-11-11 16:04:15.275 Session state does not function when running a script without `streamlit run`


In [16]:
%%writefile app.py

import streamlit as st
import re
import pickle

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences


# Load model and files
model = load_model("researchly_model.keras")

with open("tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

with open("encoder.pkl", "rb") as file:
    encoder = pickle.load(file)

with open("max_len.pkl", "rb") as file:
    max_len = pickle.load(file)


# Preprocess the abstract
def preprocess_text(text):

    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()

    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)

    text = " ".join(
        word for word in text.split()
        if word not in stop_words
    )

    text = " ".join(
        lemmatizer.lemmatize(word)
        for word in text.split()
    )

    sequence = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        sequence,
        maxlen=max_len
    )

    return padded


# Heading
st.title("Researchly Research Abstract Classifier")

st.write(
    "This app classifies research abstracts as either "
    "AI related or classical research."
)


# Input
abstract = st.text_area("Enter research abstract:")


# Button
if st.button("Classify Abstract"):

    if abstract.strip() == "":
        st.warning("Please enter a research abstract.")

    else:
        processed_text = preprocess_text(abstract)

        prediction = model.predict(
            processed_text,
            verbose=0
        )

        predicted_class = int(prediction[0][0] >= 0.5)

        predicted_label = encoder.inverse_transform(
            [predicted_class]
        )[0]

        if predicted_label == "ai":
            predicted_label = "artificial intelligence"
        else:
            predicted_label = "classical research"

        st.subheader("Prediction")

        st.write(
            "The abstract is classified as:",
            predicted_label
        )

Overwriting app.py


In [ ]:
!streamlit run app.py